# ESM-DMS Analysis Notebook

Interactive wrapper around `mega_analysis.py`. Each section exposes a simple function call for one step of the pipeline:

1. **Config** — load and resolve paths for a dataset
2. **Prepare embeddings** — convert raw embedding pkl to inference/simulation format
3. **Run inference** — infer selection coefficients from real data
4. **Run simulation** — simulate evolution and infer from synthetic data
5. **Analyses** — one cell per analysis flag

Set `DATASET` and `OUTPUT_DIR` in the Config cell, then run top-to-bottom.

## 0. Imports

In [ ]:
import os, sys

pwd = "/net/dali/home/barton/dhw28/popDMS/esmDMS"
if not os.path.exists(pwd):
    pwd = "/Users/dylanwells/popDMS/esmDMS"
if pwd not in sys.path:
    sys.path.insert(0, pwd)

import matplotlib
matplotlib.use("inline")   # show plots inside the notebook
import matplotlib.pyplot as plt

from mega_analysis import (
    load_config,
    _detect_layers,
    emb_df_to_inference_dfs,
    emb_df_to_sim_dfs,
    run_inference,
    run_simulation,
    popDMS_esmDMS_comparison_analysis,
    popDMS_enrichment_comparison_analysis,
    plot_cross_replicate_consistency_analysis,
    plot_shuffled_frequencies_analysis,
)

from analysis_helpers import (
    plot_true_vs_inferred_fitness,
    plot_true_vs_inferred_sel_coeffs,
    plot_cross_replicate_consistency,
    plot_fitness_trajectories,
    get_individual_fitness_values,
    get_esm_individual_fitness_values,
    get_enrichment_ratios,
    plot_baseline_vs_esm_comparison,
    plot_scatter_comparison,
)

print("Imports OK")

## 1. Config

Set the dataset name and output directory here. Everything else is derived from the config files.

In [ ]:
DATASET    = "Ube4b"
OUTPUT_DIR = "plots/"

EMB_CONFIG_PATH = "configs/inference_config.json"
SIM_CONFIG_PATH = "configs/simulation_config.json"

# Normalization applied to embeddings before inference
# Options: "none", "by_layer" (global z-score), "by_layer_dim" (per-dim z-score)
NORMALIZE  = "none"

# Restrict to a subset of replicates, or None to use all
REPLICATES = None   # e.g. [1, 2, 3]

# Force re-running inference even if a cache file exists
FORCE_RECOMPUTE = False

# ── Resolve configs ──────────────────────────────────────────────────────────
emb_cfg = load_config(EMB_CONFIG_PATH, DATASET)
sim_cfg = load_config(SIM_CONFIG_PATH, DATASET)

emb_cfg["normalize"]  = NORMALIZE
emb_cfg["replicates"] = REPLICATES

EMBEDDING_PATH = emb_cfg.get("embedding_path") or sim_cfg.get("embedding_path")
PATHS = {DATASET: EMBEDDING_PATH}

print(f"Dataset:        {DATASET}")
print(f"Embedding path: {EMBEDDING_PATH}")
print(f"Output dir:     {OUTPUT_DIR}")
print(f"Normalize:      {NORMALIZE}")
print(f"Replicates:     {REPLICATES}")

## 2. Prepare embeddings

Convert the raw `{dataset}_embeddings.pkl` into the compact per-layer files needed by inference and simulation.
Skip this cell if the layer files already exist.

In [ ]:
import pandas as pd

def prepare_embeddings(embedding_path, modes=("inference", "simulation")):
    """Convert raw embeddings pkl to compact inference and/or simulation formats.

    modes: iterable of 'inference' and/or 'simulation'
    """
    basename  = os.path.basename(embedding_path.rstrip("/"))
    raw_path  = os.path.join(embedding_path, f"{basename}_embeddings.pkl")

    if not os.path.exists(raw_path):
        print(f"Raw embeddings not found at {raw_path}")
        return

    # Check whether layer files already exist before loading the (large) pkl
    inf_layers = _detect_layers(embedding_path)          # compact format
    sim_layers = _detect_layers(embedding_path, "sim_df") # sim format

    need_inf = "inference"  in modes and len(inf_layers) == 0
    need_sim = "simulation" in modes and len(sim_layers) == 0

    if not need_inf and not need_sim:
        print("Layer files already exist — nothing to do.")
        return

    print(f"Loading {raw_path} ...")
    emb_df = pd.read_pickle(raw_path)
    print(f"  {len(emb_df):,} rows, {emb_df['ProteinSequence'].nunique():,} unique sequences")

    if need_inf:
        print("\nWriting compact inference format ...")
        emb_df_to_inference_dfs(emb_df, embedding_path)

    if need_sim:
        print("\nWriting simulation format ...")
        emb_df_to_sim_dfs(emb_df, embedding_path)


prepare_embeddings(EMBEDDING_PATH, modes=("inference", "simulation"))

## 3. Run inference (real data)

Infers selection coefficients $\mathbf{s}$ from observed frequency trajectories.
Results are cached to `inference_results.pkl` — re-running this cell is fast if the cache is valid.

In [ ]:
all_results_emb = {DATASET: run_inference(
    EMBEDDING_PATH, emb_cfg,
    save_results=True,
    force_recompute=FORCE_RECOMPUTE,
)}

# Quick summary
processed = all_results_emb[DATASET][2]
layers    = sorted(processed.keys())
n_reps    = processed[layers[0]][0].shape[0]
emb_dim   = processed[layers[0]][0].shape[1]
print(f"\nInference done: {len(layers)} layers, {n_reps} replicates, embedding dim {emb_dim}")

## 4. Run simulation (synthetic data)

Runs Wright-Fisher simulation on the ESM embeddings with a randomly drawn selection coefficient vector,
then infers $\mathbf{s}$ from the simulated trajectories. Used to benchmark inference accuracy.

In [ ]:
all_results_sim = {DATASET: run_simulation(EMBEDDING_PATH, sim_cfg)}

sim_layers = sorted(all_results_sim[DATASET][2].keys())
print(f"Simulation done: {len(sim_layers)} layers")

## 5. Analyses

Each cell below runs one analysis.  
Simulation analyses require `all_results_sim` (cell 4).  
Embedding analyses require `all_results_emb` (cell 3).

Set `output_dir` per-cell or leave it to derive from the globals.

In [ ]:
# Helper so plot output dirs are consistent with the CLI
import os

def emb_out(subdir=""):
    """Output directory for embedding-based analyses."""
    parts = [OUTPUT_DIR, DATASET, NORMALIZE]
    if REPLICATES:
        parts.append("reps" + "_".join(map(str, sorted(REPLICATES))))
    if subdir:
        parts.append(subdir)
    path = os.path.join(*parts)
    os.makedirs(path, exist_ok=True)
    return path

def sim_out(subdir=""):
    """Output directory for simulation-based analyses."""
    path = os.path.join(OUTPUT_DIR, DATASET, subdir) if subdir else os.path.join(OUTPUT_DIR, DATASET)
    os.makedirs(path, exist_ok=True)
    return path

### 5a. True vs inferred fitness (simulation)

In [ ]:
plot_true_vs_inferred_fitness(
    all_results_sim, PATHS, sim_cfg,
    output_dir=sim_out(),
)

### 5b. True vs inferred selection coefficients (simulation)

In [ ]:
plot_true_vs_inferred_sel_coeffs(
    all_results_sim, PATHS, sim_cfg,
    output_dir=sim_out(),
)

### 5c. Mean fitness trajectories (simulation)

In [ ]:
plot_fitness_trajectories(
    all_results_sim,
    output_dir=sim_out(),
)

### 5d. Cross-replicate consistency (real data)

In [ ]:
plot_cross_replicate_consistency_analysis(
    all_results_emb, PATHS, emb_cfg,
    output_dir=emb_out(),
)

### 5e. Cross-replicate consistency — shuffled frequencies (negative control)

In [ ]:
plot_shuffled_frequencies_analysis(
    all_results_emb, PATHS, emb_cfg,
    output_dir=emb_out(),
)

### 5f. ESM-DMS vs popDMS fitness + enrichment ratio

In [ ]:
popDMS_esmDMS_comparison_analysis(
    all_results_emb, PATHS, emb_cfg,
    output_dir=emb_out(),
)

### 5g. popDMS fitness vs enrichment ratio (direct comparison)

In [ ]:
popDMS_enrichment_comparison_analysis(
    all_results_emb, PATHS, emb_cfg,
    output_dir=emb_out(),
)

## 6. Ad-hoc exploration

Access the raw inference results for custom plots.

In [ ]:
# Unpack a single layer
LAYER = 15

processed  = all_results_emb[DATASET][2]
s          = processed[LAYER][0]       # (n_reps, emb_dim)  per-replicate s
s_joint    = processed[LAYER][1]       # (emb_dim,)         joint s
error_bars = processed[LAYER][2]       # (n_reps, emb_dim)
icov       = processed[LAYER][4]       # list of (emb_dim, emb_dim) per rep
gamma_opt  = processed[LAYER][5]       # float

print(f"Layer {LAYER}")
print(f"  s shape:      {s.shape}")
print(f"  s_joint norm: {float((s_joint**2).sum())**0.5:.4f}")
print(f"  gamma_opt:    {gamma_opt:.4g}")

In [ ]:
# Per-sequence ESM-DMS fitness at a chosen layer
esm_fits = get_esm_individual_fitness_values(
    EMBEDDING_PATH, LAYER, s_joint,
    fitness_fn=emb_cfg.get("fitness_fn", "plus1"),
    normalize=NORMALIZE,
)
print(f"{len(esm_fits):,} sequences | mean fitness: {esm_fits.mean():.4f}")
esm_fits.hist(bins=60)
plt.title(f"ESM-DMS fitness distribution — layer {LAYER}")
plt.xlabel("Fitness")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Log enrichment ratios
enrichment = get_enrichment_ratios(EMBEDDING_PATH)
print(f"{len(enrichment):,} sequences | mean log-enrichment: {enrichment.mean():.4f}")
enrichment.hist(bins=60)
plt.title("Log enrichment ratio distribution")
plt.xlabel("log(post / pre)")
plt.ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# Pearson r between ESM-DMS and enrichment across all inferred layers
from scipy.stats import pearsonr
from analysis_helpers import _align_pair
import numpy as np

layers_to_check = sorted(processed.keys())
rs = []
for lyr in layers_to_check:
    s_j = processed[lyr][1]
    esm = get_esm_individual_fitness_values(
        EMBEDDING_PATH, lyr, s_j,
        fitness_fn=emb_cfg.get("fitness_fn", "plus1"),
        normalize=NORMALIZE,
    )
    x, y = _align_pair(enrichment, esm)
    rs.append(pearsonr(x, y)[0] if len(x) >= 3 else float("nan"))

plt.figure(figsize=(8, 4))
plt.plot(layers_to_check, rs, marker="o", markersize=4, linewidth=2)
plt.axhline(0, color="gray", linestyle=":")
plt.xlabel("ESM-2 Layer")
plt.ylabel("Pearson r")
plt.title(f"ESM-DMS vs enrichment ratio — {DATASET}")
plt.tight_layout()
plt.show()

best = layers_to_check[int(np.nanargmax(rs))]
print(f"Best layer: {best}  (r = {rs[layers_to_check.index(best)]:.3f})")